In [ ]:
!pip uninstall -y mediapipe
!pip install mediapipe

Found existing installation: mediapipe 0.10.14
Uninstalling mediapipe-0.10.14:
  Successfully uninstalled mediapipe-0.10.14
  Using cached mediapipe-0.10.32-py3-none-manylinux_2_28_x86_64.whl.metadata (9.8 kB)
Using cached mediapipe-0.10.32-py3-none-manylinux_2_28_x86_64.whl (10.3 MB)


In [ ]:
import mediapipe as mp
print(mp.__version__)
print(dir(mp))

0.10.32
['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


In [ ]:
!pip uninstall -y mediapipe
!pip install mediapipe==0.10.14 opencv-python numpy

Found existing installation: mediapipe 0.10.32
Uninstalling mediapipe-0.10.32:
  Successfully uninstalled mediapipe-0.10.32
  Using cached mediapipe-0.10.14-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.7 kB)
Using cached mediapipe-0.10.14-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (35.7 MB)


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import math
from IPython.display import HTML
from base64 import b64encode
import os

# ---------------- INIT ----------------
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

mp_draw = mp.solutions.drawing_utils
drawing_spec = mp_draw.DrawingSpec(color=(0, 255, 255), thickness=1)

# --- COLAB CHANGE: Read from file instead of webcam ---
input_video = '/content/WIN_20260313_13_07_40_Pro.mp4' # Make sure you upload a file with this name
output_video = 'output.mp4'

cap = cv2.VideoCapture(input_video)

# Get video properties to set up the VideoWriter
fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Since we merge two frames side-by-side, the output width is double
out_width = w * 2

# --- COLAB CHANGE: Use VideoWriter to save output instead of showing it live ---
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (out_width, h))

blink_count = 0
blink_state = False

# Landmarks mapping
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
UPPER_LIP = 13
LOWER_LIP = 14
MOUTH_LEFT = 78
MOUTH_RIGHT = 308
NOSE_TIP = 1
LEFT_CHEEK = 234
RIGHT_CHEEK = 454

def dist(p1, p2):
    return math.dist(p1, p2)

print("Processing video... This may take a moment depending on the video length.")

# ---------------- LOOP ----------------
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break # End of video

    # Create BLACK mesh canvas
    mesh_canvas = np.zeros((h, w, 3), dtype=np.uint8)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)

    face_status = "NO"
    mouth_status = "-"
    head_status = "-"

    if results.multi_face_landmarks:
        face_status = "YES"
        landmarks = results.multi_face_landmarks[0]

        mp_draw.draw_landmarks(
            mesh_canvas,
            landmarks,
            mp_face_mesh.FACEMESH_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=drawing_spec
        )

        pts = [(int(l.x * w), int(l.y * h)) for l in landmarks.landmark]

        # -------- BLINK --------
        def ear(eye):
            v1 = dist(pts[eye[1]], pts[eye[5]])
            v2 = dist(pts[eye[2]], pts[eye[4]])
            h1 = dist(pts[eye[0]], pts[eye[3]])
            return (v1 + v2) / (2.0 * h1) if h1 > 0 else 0

        avg_ear = (ear(LEFT_EYE) + ear(RIGHT_EYE)) / 2.0

        if avg_ear < 0.20:
            if not blink_state:
                blink_count += 1
                blink_state = True
        else:
            blink_state = False

        # -------- MOUTH --------
        mouth_height = dist(pts[UPPER_LIP], pts[LOWER_LIP])
        mouth_width = dist(pts[MOUTH_LEFT], pts[MOUTH_RIGHT])
        mar = mouth_height / mouth_width if mouth_width > 0 else 0
        mouth_status = "OPEN" if mar > 0.15 else "CLOSED"

        # -------- HEAD --------
        nose_to_left = dist(pts[NOSE_TIP], pts[LEFT_CHEEK])
        nose_to_right = dist(pts[NOSE_TIP], pts[RIGHT_CHEEK])

        if nose_to_right > 0:
            head_ratio = nose_to_left / nose_to_right
            if head_ratio > 1.5:
                head_status = "RIGHT"
            elif head_ratio < 0.66:
                head_status = "LEFT"
            else:
                head_status = "CENTER"

    # -------- MERGE LEFT + RIGHT --------
    combined = np.hstack((frame, mesh_canvas))

    # -------- LABEL PANEL --------
    x_offset = w + 20
    cv2.putText(combined, f"Face Detected: {face_status}", (x_offset, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.putText(combined, f"Blink Count: {blink_count}", (x_offset, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    cv2.putText(combined, f"Mouth: {mouth_status}", (x_offset, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(combined, f"Head: {head_status}", (x_offset, 130),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 200, 0), 2)

    # --- COLAB CHANGE: Write frame to file instead of showing it ---
    out.write(combined)

cap.release()
out.release()
print("Processing complete!")

# --- COLAB CHANGE: Compress and display the video inline ---
# Browsers require H264 encoding to play MP4s properly via HTML5
os.system(f"ffmpeg -i {output_video} -vcodec libx264 compressed_output.mp4 -y -loglevel quiet")

mp4 = open('compressed_output.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

display(HTML("""
<video width=800 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url))

Processing video... This may take a moment depending on the video length.


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing complete!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
